In [6]:
import os
import wrds
import pandas as pd

db = wrds.Connection()

Loading library list...
Done


# Simple version

Factor Construction:

factors["MKT_RF"] = log_ret["SPY"] 

factors["SMB"]    = log_ret["IWM"] - log_ret["VV"]

factors["HML"]    = log_ret["VTV"] - log_ret["VUG"]

factors["MOM"]    = log_ret["MTUM"]

In [1]:

import pandas as pd


def query_minute_price_single_year(company, start_date, end_date):
    start_year = start_date[:4]
    end_year = end_date[:4]

    if start_year != end_year:
        raise ValueError("start_date and end_date should be in the same year.")

    schema = f"taqm_{start_year}.ctm_{start_year}"

    sql = f"""
    SELECT
        TO_CHAR(date, 'YYYY-MM-DD') || ' ' || SUBSTR(time_m::VARCHAR, 1, 5) AS datetime,
        price AS last_trade_price,
        date,
        SUBSTR(time_m::VARCHAR, 1, 5) AS time_hm
    FROM (
        SELECT *,
               RANK() OVER (
                   PARTITION BY date, sym_root, EXTRACT(HOUR FROM time_m), EXTRACT(MINUTE FROM time_m)
                   ORDER BY time_m DESC, time_m_nano DESC
               ) AS rnk
        FROM {schema}
        WHERE sym_root = '{company}'
          AND date BETWEEN '{start_date}' AND '{end_date}'
    ) a
    WHERE rnk = 1
    ORDER BY date, EXTRACT(HOUR FROM time_m), EXTRACT(MINUTE FROM time_m);
    """

    df = db.raw_sql(sql)
    return df.sort_values(by="datetime").reset_index(drop=True)


In [ ]:
import pathlib, datetime as dt, wrds, pandas as pd

OUTDIR = pathlib.Path("./minute_prices_etf")
OUTDIR.mkdir(exist_ok=True)

ETFS        = ["SPY", "IWM", "VV", "VTV", "VUG", "MTUM"]
START, END  = "2020-11-01", "2021-04-30"
s_date, e_date = map(lambda x: dt.datetime.strptime(x, "%Y-%m-%d"), [START, END])

with wrds.Connection() as db_conn:     # Open connection once
    db = db_conn                       # Assign to global variable for function use

    for tic in ETFS:
        pieces = []
        for yr in range(s_date.year, e_date.year + 1):
            yr_start = START if yr == s_date.year else f"{yr}-01-01"
            yr_end   = END   if yr == e_date.year else f"{yr}-12-31"
            pieces.append(query_minute_price_single_year(tic, yr_start, yr_end))

        df_all = (pd.concat(pieces, ignore_index=True)
                    .drop_duplicates("datetime", keep="last")   # Remove duplicates
                    .sort_values("datetime")
                    .reset_index(drop=True))

        csv_name = OUTDIR / f"{tic}_{START.replace('-','')}-{END.replace('-','')}_minute_price.csv"
        df_all.to_csv(csv_name, index=False)
        print(f"✓ {tic}: saved {len(df_all):,} rows → {csv_name.name}")

print("\nAll ETF minute-price files are ready in", OUTDIR.resolve())

Loading library list...
Done
✓ SPY: saved 113,792 rows → SPY_20201101-20210430_minute_price.csv
✓ IWM: saved 95,957 rows → IWM_20201101-20210430_minute_price.csv
✓ VV: saved 46,717 rows → VV_20201101-20210430_minute_price.csv
✓ VTV: saved 52,460 rows → VTV_20201101-20210430_minute_price.csv
✓ VUG: saved 52,746 rows → VUG_20201101-20210430_minute_price.csv
✓ MTUM: saved 50,030 rows → MTUM_20201101-20210430_minute_price.csv

All ETF minute-price files are ready in /Users/zhaomufan/学习资料/wrds project/ff-factor-minute/minute_prices_etf


In [ ]:
# ---------------------------------------------------------------------------
# 0. imports & paths  —— Keep unchanged
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, pathlib, wrds

DATA_DIR = pathlib.Path("./minute_prices_etf")
OUT_FILE = "ff_factors_20201101_20210430_minute.csv"
CSV_MAP  = {
    "SPY": "SPY_20201101-20210430_minute_price.csv",
    "IWM": "IWM_20201101-20210430_minute_price.csv",
    "VV" : "VV_20201101-20210430_minute_price.csv",
    "VTV": "VTV_20201101-20210430_minute_price.csv",
    "VUG": "VUG_20201101-20210430_minute_price.csv",
    "MTUM":"MTUM_20201101-20210430_minute_price.csv",
}
MINUTES_PER_DAY = 390   # 09:30–16:00

# ---------------------------------------------------------------------------
# 1. read CSVs → wide price table
# ---------------------------------------------------------------------------
price_df = pd.DataFrame()
for tic, fn in CSV_MAP.items():
    s = (
        pd.read_csv(DATA_DIR / fn, parse_dates=["datetime"])
          .set_index("datetime")["last_trade_price"]
          .rename(tic)
    )
    price_df = price_df.join(s, how="outer") if not price_df.empty else s.to_frame()

price_df = price_df.sort_index()

# ☆ Keep only regular trading hours 09:30–16:00
price_df = price_df.between_time("09:30", "16:00")

# ---------------------------------------------------------------------------
# 2. minute log returns  (Drop any minutes containing NaN)
# ---------------------------------------------------------------------------
log_ret = np.log(price_df / price_df.shift(1)).dropna(how="any")

# ---------------------------------------------------------------------------
# 3. pull daily RF & split to minute level
# ---------------------------------------------------------------------------
# with wrds.Connection() as db:
#     rf_daily = db.raw_sql("""
#         SELECT date, rf
#         FROM   ff.factors_daily
#         WHERE  date BETWEEN '2020-11-01' AND '2021-04-30'
#     """).set_index("date")["rf"]

# rf_minute = log_ret.index.normalize().map(rf_daily) / MINUTES_PER_DAY

# ---------------------------------------------------------------------------
# 4. build factors
# ---------------------------------------------------------------------------
factors = pd.DataFrame(index=log_ret.index)
factors["MKT_RF"] = log_ret["SPY"]  
factors["SMB"]    = log_ret["IWM"]  - log_ret["VV"]
factors["HML"]    = log_ret["VTV"]  - log_ret["VUG"]
factors["MOM"]    = log_ret["MTUM"] 
# ---------------------------------------------------------------------------
# 5. save & preview
# ---------------------------------------------------------------------------
factors.to_csv(OUT_FILE)
print("saved →", OUT_FILE, " |  rows:", len(factors))
factors.head()


saved → ff_factors_20201101_20210430_minute.csv  |  rows: 43084


,MKT_RF,SMB,HML,MOM
datetime,,,,
2020-11-02 09:31:00,-0.001092,-0.000641,0.000508,-0.001546
2020-11-02 09:32:00,0.001304,-0.000361,-0.001074,0.002072
2020-11-02 09:33:00,-0.000454,0.000130,0.001523,-0.001430
2020-11-02 09:34:00,-0.000819,-0.001114,-0.002327,0.000272
2020-11-02 09:35:00,0.000788,0.001859,-0.000183,0.001334
